# HydroFragments quickstart

**What this notebook does:** runs the full HydroFragments pipeline once, end to end, on a tiny built-in example, so you can see the shape of the input, the shape of the output, and one plot -- in under two minutes, no external data download required.

**What HydroFragments measures:** how much of a river reach is holding water over time, and how fragmented that water is into separate pools. It does this from a time series of water/no-water maps (a "water mask"), not from streamflow or depth data.

**What this notebook is not:** a real analysis. The data below is synthetic (a square of "water" that shrinks every month) so the story is obvious and the numbers are easy to sanity-check by eye. For a real Digital Earth Australia (DEA) workflow, see `02_dea_via_tsfill.ipynb`. For a tour of what each metric family means, see `03_metrics_walkthrough.ipynb`.

## 1. A tiny example water mask

HydroFragments needs a time series of water masks -- one true/false grid per date, all on the same grid. Here we build a synthetic 4-month series: an 11x11 pixel patch of "water" that shrinks from a 7x7 square down to a single pixel, one ring smaller each month. Think of it as a pool drying down over a season.

This comes from a small helper bundled with the examples (`examples/_fixtures.py`) so the exact same fixture is also checked by the test suite -- what you see below is guaranteed to be what actually runs in CI.

In [ ]:
import sys
from pathlib import Path

# Make the bundled example fixture importable when running this notebook
# directly from the examples/ directory.
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt

from _fixtures import quickstart_water_timeseries, quickstart_config

water = quickstart_water_timeseries()
water

Each `time` slice is an 11x11 grid of `True`/`False` (water/no-water). Let's look at the wet-pixel count per month -- it should shrink 49, 25, 9, 1, matching a reach that is visibly drying down.

In [ ]:
wet_pixel_counts = water.sum(dim=("y", "x"))
wet_pixel_counts.to_series()

## 2. Open it as a HydroFragments `WaterCube`

`open_water_cube` is the single entry point for turning a raw array/Dataset/file path into the canonical `(water, valid_obs)` pair HydroFragments works with. Our fixture is an already-boolean array with no separate valid-observation layer, so we pass `input_kind="generic_binary"` explicitly (auto-detection -- leaving `input_kind=None` -- would land on the same adapter here; being explicit just makes this walkthrough easier to follow).

With no `valid_obs` supplied, every pixel is treated as validly observed -- fine for this synthetic example, but real satellite-derived masks almost always carry a genuine valid-observation layer (cloud/shadow/no-data masking). See `02_dea_via_tsfill.ipynb` for that case.

In [ ]:
from hydrofragments import open_water_cube

cube = open_water_cube(water, input_kind="generic_binary")
cube.provenance

## 3. Configure and run `analyze`

`analyze` needs a `HydroConfig` describing scientific settings -- what kind of input this is, how the time series was composited into monthly steps, where to write output. `HydroConfig.from_mapping` builds one from a plain dict (this is exactly what a YAML config file loads into).

Our fixture is already one frame per calendar month, so `monthly_composite: "supplied"` / `composite_owner: "caller"` tell HydroFragments there is nothing left for it to composite.

In [ ]:
from hydrofragments import HydroConfig, analyze

config = HydroConfig.from_mapping(quickstart_config(output_dir="quickstart_out"))
result = analyze(cube, aoi_id="quickstart_demo", config=config, pixel_size_m=30.0)

result.run_id, result.output_dir

## 4. Read the tidy metrics table

`result.metrics_table` is a tidy `pandas.DataFrame`: one row per metric/date/statistic. Every metric family lives in the same table, distinguished by the `metric` and `metric_family` columns -- there is no separate output shape per metric.

In [ ]:
metrics = result.metrics_table
metrics[["date", "metric", "metric_family", "value", "unit"]].sort_values(
    ["metric", "date"]
).reset_index(drop=True)

A couple of things worth noticing before moving on:

- **`is_reportable`** marks whether a value should actually be trusted/plotted -- a metric can be present in the table but flagged not-reportable (e.g. too few valid observations that month). Always filter on it before using a value.
- If your real data has low satellite coverage, `analyze` may add a warning to the run manifest recommending you pre-process with WaterMask-TSFill (a separate gapfilling tool) before running HydroFragments -- HydroFragments itself never gapfills or gap-interpolates. Our synthetic fixture has full coverage, so no such warning is expected here, but keep an eye out for it on real data (see `result.manifest` and `docs/input_format.md`).

## 5. One plot

APSEC ("wetted area fraction of the fixed reach") is a good first metric to plot: it directly tracks how much of the reach is holding water each month. We expect it to fall in step with the shrinking wet-pixel counts from Section 1.

In [ ]:
apsec = metrics[metrics["metric"] == "apsec"].sort_values("date")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(apsec["date"], apsec["value"], marker="o")
ax.set_ylabel("APSEC (% of reach wetted)")
ax.set_xlabel("month")
ax.set_title("Wetted area fraction over the synthetic dry-down")
fig.autofmt_xdate()
plt.show()

## Next steps

- **Real DEA data:** `02_dea_via_tsfill.ipynb` walks through pointing `open_water_cube` at an actual WaterMask-TSFill output instead of a synthetic fixture.
- **Understanding every metric family:** `03_metrics_walkthrough.ipynb` runs through Extent & Persistence, Morphology & Fragmentation, Clustering & Connectivity, and Dynamics one section at a time, plus the 4-zone spatial stratification (`build_zones`).
- **Command line:** once you have a real config file and input, the same pipeline is available without opening a notebook:
  ```bash
  hydrofragments analyze --config cfg.yaml --input data.zarr --aoi my_reach --out results/
  ```
- **Input format details:** see `docs/input_format.md` for the full adapter contract (what shapes/sentinel values each `input_kind` expects).